In [2]:
# Cell 1: Check Python version and GPU availability
import sys
import platform

print(f"Python:   {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"PyTorch:  {torch.__version__}")
    print(f"CUDA:     {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")
except ImportError:
    print("PyTorch:  NOT INSTALLED — run Cell 2 first")

are we ready?


In [4]:
# Cell 2: Install ML dependencies (run once)
# These match backend/requirements.txt ML section

%pip install torch>=2.2.0 transformers>=4.44.0 sentencepiece>=0.2.0
%pip install accelerate>=0.30.0 peft>=0.11.0 bitsandbytes>=0.43.0

ERROR: Could not find a version that satisfies the requirement request (from versions: none)
ERROR: No matching distribution found for request
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 64.8 MB/s eta 0:00:0000:0100:01


In [6]:
# Cell 3: Verify all imports work
import torch
import transformers
import sentencepiece
import accelerate
import peft

print("All imports successful!")
print(f"  torch:         {torch.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  accelerate:    {accelerate.__version__}")

# PUENTE — Quick Start: Environment Setup

Install all required Python packages for NLLB-200 inference and LoRA fine-tuning.

**Target Hardware:** Ryzen 5 / i5, 8GB RAM, Windows 11 + WSL2

In [7]:
# Cell 4: Download NLLB-200 model (one-time, ~2.4GB download)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os

model_name = "facebook/nllb-200-distilled-600M"
save_path = "../ml_models/nllb-200-distilled-600M"

if os.path.exists(save_path):
    print(f"Model already exists at {save_path}")
else:
    print(f"Downloading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    tokenizer.save_pretrained(save_path)
    model.save_pretrained(save_path)
    print(f"Saved to {save_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 23.3 MB/s eta 0:00:00


In [8]:
# Cell 5: Quick translation test
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

save_path = "../ml_models/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSeq2SeqLM.from_pretrained(save_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

# Test: English → Chavacano
tokenizer.src_lang = "eng_Latn"
inputs = tokenizer("Good morning, how are you?", return_tensors="pt").to(device)
tgt_id = tokenizer.convert_tokens_to_ids("cbk_Latn")

with torch.no_grad():
    output = model.generate(**inputs, forced_bos_token_id=tgt_id, max_new_tokens=128, num_beams=4)

result = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(f"English → Chavacano: {result}")
print("\nSetup complete! You can now run the Django backend with NLLB-200.")

Looking in links: https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1.0-cp310-cp310-linux_x86_64.whl
ERROR: Could not find a version that satisfies the requirement torch~=2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch~=2.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 89.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.6/208.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.9/437.9 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.8 MB/s eta 0:00:00
